# Chapter 6 — Fine-Tuning Gemma 4 with TRL

The complete flow from Chapter 6: pull the synthetic medical Q&A dataset, convert it to
prompt-completion form, split it, validate and inspect it, then run TRL's `SFTTrainer`
against **Gemma 4 E4B** ("effective 4B": 8B total parameters, ~4.5B effective via
per-layer embeddings).

**Hardware:** a full fine-tune of an 8B model wants an A100/H100-class GPU. The optional
LoRA cell below drops the requirement to a single 24–48GB GPU (or a Colab A100). For a
pipeline smoke test on smaller hardware, set `max_steps=20` in the config cell.

**Access:** Gemma models are gated — accept the license at
[google/gemma-4-E4B-it](https://huggingface.co/google/gemma-4-E4B-it) and log in below.

In [ ]:
%pip install -U "datasets>=5.0" transformers trl accelerate peft torchao -q
# torchao is included because an outdated copy (common in shared
# environments) causes an ImportError when transformers loads the model.
# If this cell upgrades anything, restart the kernel before continuing.

In [ ]:
# Gemma is gated on the Hugging Face Hub — log in with a token that has
# accepted the license on the model page.
from huggingface_hub import notebook_login
notebook_login()

## 1. Load the tokenizer and peek at the chat template

Before anything else, look at what the template produces (`tokenize=False` is the
debugging trick from the chapter). Note that Gemma 4 accepts a standalone `system`
role — earlier Gemma generations did not.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "google/gemma-4-E4B-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

demo_messages = [
    {"role": "system", "content": "You are a helpful medical assistant."},
    {"role": "user", "content": "What are the warning signs of high blood pressure?"},
    {"role": "assistant", "content": "High blood pressure rarely causes noticeable symptoms..."},
]

print(tokenizer.apply_chat_template(demo_messages, tokenize=False,
                                    add_generation_prompt=False))

## 2. Load the dataset and convert to prompt-completion form

The hosted dataset ([lmoroney/synthdata](https://huggingface.co/datasets/lmoroney/synthdata))
is in Alpaca format. We convert each record so the `prompt` holds everything the model
reads as context (system + user turns) and the `completion` holds only the assistant's
answer. Paired with `completion_only_loss=True` in the trainer config, this gives us the
loss masking described in the chapter: the gradient comes only from the answer tokens.

In [ ]:
from datasets import load_dataset

SYSTEM_PROMPT = """You are a board-certified physician providing educational \
medical information. Your answers are accurate, practical, and always recommend \
consulting a healthcare provider for personalized advice."""


def to_prompt_completion(example: dict) -> dict:
    user_content = example["instruction"]
    if example.get("input", "").strip():
        user_content = f"{example['instruction']}\n\n{example['input']}"
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
        ],
        "completion": [
            {"role": "assistant", "content": example["output"]},
        ],
    }


raw = load_dataset("lmoroney/synthdata", split="train")
dataset = raw.map(to_prompt_completion, remove_columns=raw.column_names)
print(dataset)

## 3. Split 85/10/5 into train / validation / test

The validation set monitors loss during training; the test set stays untouched until
after training for an honest generalization estimate.

In [ ]:
# Extract 85% for training, leaving 15% as a remainder
train_and_remainder = dataset.train_test_split(test_size=0.15, seed=42)
train_set = train_and_remainder["train"]
remainder = train_and_remainder["test"]

# Split the 15% remainder: 10% validation, 5% test (5% is 1/3 of 15%)
val_test_split = remainder.train_test_split(test_size=1 / 3, seed=42)
val_set = val_test_split["train"]
test_set = val_test_split["test"]

print(f"train: {len(train_set)}, validation: {len(val_set)}, test: {len(test_set)}")

## 4. Validate: token-length statistics

A large gap between p90 and p99 flags a handful of unusually long examples. Anything
over the max length will be truncated in training — for answers, that means learning
to stop mid-sentence, so check this before training, not after.

In [ ]:
MAX_TOKENS = 2048

lengths = []
for example in train_set:
    formatted = tokenizer.apply_chat_template(
        example["prompt"] + example["completion"],
        tokenize=False, add_generation_prompt=False,
    )
    lengths.append(len(tokenizer.encode(formatted)))

s = sorted(lengths)
pick = lambda p: s[min(int(len(s) * p), len(s) - 1)]
over = sum(1 for n in lengths if n > MAX_TOKENS)
print(f"Token length — p50: {pick(0.50)}, p90: {pick(0.90)}, "
      f"p99: {pick(0.99)}, max: {s[-1]}")
print(f"Over limit ({MAX_TOKENS} tokens): {over}")

## 4b. Validate: format rules

Beyond length, every example should pass structural checks before training: the right
keys, no empty messages, roles in the right order, and a chat template that renders
without throwing. These are the validation rules from the chapter's Format Validation
section (the full standalone version is `chapter06_split_and_validate.py` in the
download).

In [ ]:
def validate_example(example):
    """Returns (is_valid, list_of_issues) for one prompt-completion example."""
    issues = []
    messages = example.get("prompt", []) + example.get("completion", [])

    # The messages list should not be empty
    if not messages:
        return False, ["messages list is empty"]

    # The last message should be the model's turn
    if messages[-1]["role"] not in ("assistant", "model"):
        issues.append(f"last role is '{messages[-1]['role']}', expected the model's turn")

    # At least one message should come from the user
    if not any(m["role"] == "user" for m in messages):
        issues.append("no user message in conversation")

    # There should be no empty messages
    for i, m in enumerate(messages):
        if not m.get("content", "").strip():
            issues.append(f"message {i} (role={m.get('role')}) is empty")

    # The template must apply without throwing
    try:
        tokenizer.apply_chat_template(messages, tokenize=False)
    except Exception as e:
        issues.append(f"chat template failed: {e}")

    return len(issues) == 0, issues


valid = 0
for i, example in enumerate(train_set):
    ok, issues = validate_example(example)
    if ok:
        valid += 1
    else:
        print(f"Example {i} FAILED: {'; '.join(issues)}")

print(f"Valid examples: {valid}/{len(train_set)}")

## 5. Inspect the loss masking

Reproduce the trainer's prompt-prefix masking for one example and eyeball it. Only the
assistant's answer tokens (and the closing turn token) should print `TRAIN` — if you see
`TRAIN` on system or instruction tokens, the prompt/completion boundary is wrong.

In [ ]:
example = train_set[0]

prompt_text = tokenizer.apply_chat_template(
    example["prompt"], tokenize=False, add_generation_prompt=True)
full_text = tokenizer.apply_chat_template(
    example["prompt"] + example["completion"],
    tokenize=False, add_generation_prompt=False)

prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
full_ids = tokenizer.encode(full_text, add_special_tokens=False)
labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]

for token_id, label in zip(full_ids, labels):
    status = "TRAIN" if label != -100 else "MASK "
    print(f"{status} | {tokenizer.decode([token_id])!r}")

## 6. Load the model

`device_map="auto"` places the model across available GPU memory. The optional second
cell wraps it in a LoRA adapter (covered in depth in Chapter 7) — skip it only if you
have the hardware for a full fine-tune.

In [ ]:
import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

In [ ]:
# OPTIONAL (recommended on a single GPU): add a LoRA adapter.
# Trains ~1% of the parameters — see Chapter 7 for how this works.
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)
# If you skip this cell, set peft_config = None before the trainer cell.


## 7. Configure and run the trainer

`completion_only_loss=True` is the loss masking switch: everything in the `prompt`
column is masked to -100 and only `completion` tokens produce gradient. For a quick
pipeline smoke test, uncomment `max_steps=20`.

In [ ]:
from trl import SFTTrainer, SFTConfig

config = SFTConfig(
    output_dir="checkpoints/medical-qa-gemma4-v1",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,          # LoRA rate; use ~2e-5 for a full fine-tune
    warmup_ratio=0.05,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    bf16=True,
    max_length=2048,
    completion_only_loss=True,   # mask the prompt; train on the answer
    # max_steps=20,              # uncomment for a quick smoke test
)

trainer = SFTTrainer(
    model=model,
    args=config,
    train_dataset=train_set,
    eval_dataset=val_set,
    processing_class=tokenizer,
    peft_config=peft_config,     # None for a full fine-tune
)

trainer.train()

## 8. Try the fine-tuned model

Generate an answer for a held-out test question — same chat template, this time with
`add_generation_prompt=True` to open the assistant turn.

In [ ]:
test_example = test_set[0]
print("QUESTION:", test_example["prompt"][-1]["content"], "\n")

inputs = tokenizer.apply_chat_template(
    test_example["prompt"],
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)

with torch.no_grad():
    output = model.generate(inputs, max_new_tokens=400)

print(tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True))

## 9. Save the result

With LoRA this saves just the adapter (a few hundred MB); merge it into the base
weights later, or load it with PEFT at inference time.

In [ ]:
trainer.save_model("medical-qa-gemma4-final")
tokenizer.save_pretrained("medical-qa-gemma4-final")